In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

In [2]:
df = pd.read_csv("journal_entries_raw.csv", parse_dates=["date", "posting_datetime"])

In [3]:
df.head()

,entry_id,date,posting_time,posting_datetime,department,gl_account,account_description,vendor,amount,preparer,approver,po_number,invoice_number,po_amount,invoice_amount,payment_amount,planted_anomaly_type
0,100534,2024-01-01,09:24:00,2024-01-01 09:24:00,IT,5000,Raw Materials Expense,"Howard, Tucker and Flores",3772.81,Michael Rivas,Jeffrey Hancock,PO83001,INV635687,3772.81,3772.81,3772.81,none
1,103270,2024-01-01,10:12:00,2024-01-01 10:12:00,Sales,5300,Professional Fees,Larson PLC,14698.16,Michael Fox,Jeffrey Hancock,PO314779,INV652616,14698.16,14698.16,14698.16,none
2,100731,2024-01-01,11:05:00,2024-01-01 11:05:00,Sales,5500,Utilities,"Brown, Baker and Alvarado",14696.45,Dawn Hall,Lawrence Davis,PO881236,INV991598,14696.45,14696.45,14696.45,none
3,103381,2024-01-01,11:31:00,2024-01-01 11:31:00,Procurement,5000,Raw Materials Expense,Raymond-Lucas,36706.24,Janice Bush,Sergio Arroyo,PO88088,INV505880,36706.24,36706.24,36706.24,none
4,104584,2024-01-01,13:01:00,2024-01-01 13:01:00,Sales,5100,Office Supplies,Pearson PLC,6997.03,Michael Fox,Anita Harris,PO362999,INV98206,6997.03,6997.03,6997.03,none


In [4]:
# --- Test 1: Duplicate payment (same vendor + amount + invoice number appearing >1 time) ---
dup_mask = df.duplicated(subset=["vendor", "amount", "invoice_number"], keep=False)
df["flag_duplicate_payment"] = dup_mask

In [5]:
# --- Test 2: Three-way match mismatch (PO vs invoice vs payment amount should all agree) ---
df["flag_three_way_mismatch"] = (
    (abs(df["po_amount"] - df["invoice_amount"]) > 1) |
    (abs(df["invoice_amount"] - df["payment_amount"]) > 1)
)

In [6]:
# --- Test 3: Weekend posting ---
df["day_of_week"] = df["posting_datetime"].dt.dayofweek  # 0=Mon ... 6=Sun
df["flag_weekend_posting"] = df["day_of_week"] >= 5

In [7]:
# --- Test 4: Odd-hour posting (outside 6am-9pm business window) ---
df["hour_of_posting"] = df["posting_datetime"].dt.hour
df["flag_odd_hour_posting"] = (df["hour_of_posting"] < 6) | (df["hour_of_posting"] > 21)

In [8]:
# --- Test 5: Round-number amounts (suspiciously clean figures) ---
df["flag_round_number"] = (df["amount"] % 5000 == 0) & (df["amount"] > 0)

In [9]:
# --- Test 6: Segregation of duties violation (preparer == approver) ---
df["flag_segregation_violation"] = df["preparer"] == df["approver"]

In [10]:
rule_flag_cols = [
    "flag_duplicate_payment", "flag_three_way_mismatch", "flag_weekend_posting",
    "flag_odd_hour_posting", "flag_round_number", "flag_segregation_violation",
]
df["rule_flag_count"] = df[rule_flag_cols].sum(axis=1)

In [11]:
# Feature engineering: give the model behavioral signals, not raw IDs
df["vendor_amount_mean"] = df.groupby("vendor")["amount"].transform("mean")
df["vendor_amount_std"] = df.groupby("vendor")["amount"].transform("std").fillna(1)
df["amount_zscore_vs_vendor"] = (df["amount"] - df["vendor_amount_mean"]) / df["vendor_amount_std"].replace(0, 1)

vendor_freq = df["vendor"].value_counts()
df["vendor_txn_frequency"] = df["vendor"].map(vendor_freq)

ml_features = [
    "amount", "hour_of_posting", "day_of_week",
    "vendor_txn_frequency", "amount_zscore_vs_vendor",
]

model = IsolationForest(
    n_estimators=200,
    contamination=0.04,   # assume ~4% of population is anomalous
    random_state=42,
)
df["ml_anomaly_flag"] = model.fit_predict(df[ml_features])           # -1 = anomaly, 1 = normal
df["ml_anomaly_score"] = -model.decision_function(df[ml_features])   # higher = more anomalous
df["flag_ml_anomaly"] = df["ml_anomaly_flag"] == -1

In [12]:

# Normalize ML score to 0-1 range for a comparable composite
ml_min, ml_max = df["ml_anomaly_score"].min(), df["ml_anomaly_score"].max()
df["ml_score_normalized"] = (df["ml_anomaly_score"] - ml_min) / (ml_max - ml_min)

df["composite_risk_score"] = (
    (df["rule_flag_count"] / len(rule_flag_cols)) * 0.6 +   # rules weighted 60%
    df["ml_score_normalized"] * 0.4                          # ML weighted 40%
)

def risk_tier(row):
    if row["rule_flag_count"] >= 2 and row["flag_ml_anomaly"]:
        return "Critical"
    elif row["rule_flag_count"] >= 1 or row["flag_ml_anomaly"]:
        return "High"
    elif row["composite_risk_score"] > 0.15:
        return "Medium"
    else:
        return "Low"

df["risk_tier"] = df.apply(risk_tier, axis=1)

In [13]:
planted = df[df["planted_anomaly_type"] != "none"]
caught = planted[(planted["rule_flag_count"] > 0) | (planted["flag_ml_anomaly"])]
detection_rate = len(caught) / len(planted) * 100

false_positives = df[(df["planted_anomaly_type"] == "none") &
                      ((df["rule_flag_count"] > 0) | (df["flag_ml_anomaly"]))]
fp_rate = len(false_positives) / len(df[df["planted_anomaly_type"] == "none"]) * 100

print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Total transactions: {len(df)}")
print(f"Planted anomalies: {len(planted)}")
print(f"Caught by rules and/or ML: {len(caught)} ({detection_rate:.1f}% detection rate)")
print(f"False positives on normal transactions: {len(false_positives)} ({fp_rate:.1f}% FP rate)")
print()
print("Detection rate by planted anomaly type:")
for atype in planted["planted_anomaly_type"].unique():
    subset = planted[planted["planted_anomaly_type"] == atype]
    sub_caught = subset[(subset["rule_flag_count"] > 0) | (subset["flag_ml_anomaly"])]
    print(f"  {atype}: {len(sub_caught)}/{len(subset)} caught ({len(sub_caught)/len(subset)*100:.0f}%)")
print()
print("Risk tier distribution:")
print(df["risk_tier"].value_counts())

VALIDATION SUMMARY
Total transactions: 5047
Planted anomalies: 247
Caught by rules and/or ML: 242 (98.0% detection rate)
False positives on normal transactions: 145 (3.0% FP rate)

Detection rate by planted anomaly type:
  segregation_violation: 33/33 caught (100%)
  unusual_amount: 29/34 caught (85%)
  duplicate_payment: 94/94 caught (100%)
  weekend_posting: 25/25 caught (100%)
  three_way_mismatch: 17/17 caught (100%)
  round_number: 23/23 caught (100%)
  odd_hour_posting: 21/21 caught (100%)

Risk tier distribution:
risk_tier
Low         4501
High         385
Medium       159
Critical       2
Name: count, dtype: int64


In [14]:
export_cols = [
    "entry_id", "date", "posting_time", "department", "gl_account", "account_description",
    "vendor", "amount", "preparer", "approver", "po_number", "invoice_number",
    "flag_duplicate_payment", "flag_three_way_mismatch", "flag_weekend_posting",
    "flag_odd_hour_posting", "flag_round_number", "flag_segregation_violation",
    "rule_flag_count", "flag_ml_anomaly", "ml_anomaly_score", "composite_risk_score", "risk_tier",
]
df[export_cols].to_csv("journal_entries_scored.csv", index=False)

# Vendor-level summary (for a dashboard drill-down page)
vendor_summary = df.groupby("vendor").agg(
    total_transactions=("entry_id", "count"),
    total_amount=("amount", "sum"),
    flagged_transactions=("rule_flag_count", lambda x: (x > 0).sum()),
    avg_composite_risk=("composite_risk_score", "mean"),
).reset_index().sort_values("avg_composite_risk", ascending=False)
vendor_summary.to_csv("vendor_risk_summary.csv", index=False)

# Department-level summary
dept_summary = df.groupby("department").agg(
    total_transactions=("entry_id", "count"),
    total_amount=("amount", "sum"),
    critical_count=("risk_tier", lambda x: (x == "Critical").sum()),
    high_count=("risk_tier", lambda x: (x == "High").sum()),
).reset_index()
dept_summary.to_csv("department_risk_summary.csv", index=False)

print("\nExported: journal_entries_scored.csv, vendor_risk_summary.csv, department_risk_summary.csv")


Exported: journal_entries_scored.csv, vendor_risk_summary.csv, department_risk_summary.csv
